# Week 10 — Python Solution Lab
## Angular Momentum

**Companion to `notebooks/Week_10.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_10.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P4` | The Figure Skater | why KE rises while L is conserved |
| **L2 · Intermediate** | `P6` | Two Coaxial Disks Coupling | rotational inelastic collision, total-loss tuning |
| **L3 · Challenge** | `P9` | Turntable, Putty, and a Restoring Motor | two-phase analysis, two conservation rules |

---

## L1 · Basic — P4: The Figure Skater

> **Problem (Week_10.ipynb, L1 — P4).** A skater with $I = 4.6$ kg·m² spins at $1.5$ rev/s.
> She pulls her arms in, reducing $I$ to $1.8$ kg·m². Find her new angular velocity in rev/s.

**Diagram → Principle.** No external torque about the spin axis, so $L = I\omega$ is conserved.
Pulling the arms in is an *internal* rearrangement.

**Equation.** $I_i\omega_i = I_f\omega_f$.

**Hand prediction.** $\omega_f = 1.5 \times 4.6/1.8 = 3.83$ rev/s.

**What Python adds.** The energy bookkeeping. $L$ is conserved but $KE = L^2/2I$ is **not** — it
goes *up*, by a factor $I_i/I_f$. Where does that energy come from? The skater's muscles, doing
work to pull her arms inward against the centrifugal tendency. Computing the number turns a
"magic" demo into a closed energy account.

In [ ]:
# ═══ W10 · L1 · P4 — Conserved L, increasing KE, and who pays for it ═══
import numpy as np

# --- MODEL --------------------------------------------------------------
I_i, f_i = 4.6, 1.5        # kg*m^2, rev/s (arms out)
I_f      = 1.8             # kg*m^2       (arms in)

w_i = 2*np.pi*f_i                  # rad/s

# --- PREDICT: angular momentum conservation -----------------------------
L   = I_i * w_i
w_f = L / I_f
f_f = w_f / (2*np.pi)
print(f"L = I_i w_i = {L:.4f} kg*m^2/s   (this is the conserved quantity)")
print(f"omega_i = {w_i:.4f} rad/s  ->  omega_f = {w_f:.4f} rad/s")
print(f"f_i = {f_i:.2f} rev/s      ->  f_f = {f_f:.4f} rev/s")
print(f"speed-up factor = I_i/I_f = {I_i/I_f:.4f}")

# --- VERIFY: L really is unchanged --------------------------------------
assert np.isclose(I_i*w_i, I_f*w_f)
print(f"\ncheck: I_f w_f = {I_f*w_f:.4f} = I_i w_i  [conserved]")

# --- The part the formula hides: kinetic energy is NOT conserved --------
KE_i = 0.5*I_i*w_i**2
KE_f = 0.5*I_f*w_f**2
print(f"\nKE before = {KE_i:8.3f} J")
print(f"KE after  = {KE_f:8.3f} J")
print(f"GAIN      = {KE_f - KE_i:8.3f} J   (a factor of {KE_f/KE_i:.3f} = I_i/I_f)")

# --- Why: KE = L^2 / (2I), so shrinking I at fixed L raises KE ----------
print(f"\n  KE = L^2/(2I):  {L**2/(2*I_i):.3f} J -> {L**2/(2*I_f):.3f} J")
assert np.isclose(KE_i, L**2/(2*I_i)) and np.isclose(KE_f, L**2/(2*I_f))
assert np.isclose(KE_f/KE_i, I_i/I_f)
print("  The skater's ARMS do that work, pulling inward against the rotation.")
print("  Angular momentum is free; kinetic energy is not.")

# --- Contrast with the wrong intuition ----------------------------------
print(f"\n  If instead ENERGY were conserved, we would get")
w_wrong = np.sqrt(2*KE_i/I_f)
print(f"    omega_f = sqrt(2 KE_i / I_f) = {w_wrong/(2*np.pi):.3f} rev/s -- but that would")
print(f"    require L to jump from {L:.2f} to {I_f*w_wrong:.2f} kg*m^2/s with no torque. "
      "Impossible.")

# --- CHECK --------------------------------------------------------------
assert abs(f_f - 3.833) < 0.005
print(f"\n[OK] Matches textbook answer: omega_f = {f_f:.2f} rev/s")

## L2 · Intermediate — P6: Two Coaxial Disks Coupling

> **Problem (Week_10.ipynb, L2 — P6).** Disk A ($I_A = 0.50$ kg·m²) spins at $8.0$ rad/s.
> Disk B ($I_B = 0.30$ kg·m²) spins at $-5.0$ rad/s. They are brought together to a common
> angular velocity. Find (a) $\omega_f$ and (b) the fraction of kinetic energy lost.

**Diagram → Principle.** This is a **perfectly inelastic collision in rotational form**. The
coupling torques are internal, so $L$ is conserved; friction between the faces dissipates energy.

**Equation.** $\omega_f = \dfrac{I_A\omega_A + I_B\omega_B}{I_A + I_B}$.

**Hand prediction.** $\omega_f = (4.0 - 1.5)/0.8 = 3.125$ rad/s.
$KE_i = 19.75$ J, $KE_f = 3.906$ J, so the loss is $15.844/19.75 = 80.2\%$.

> ⚠️ **Answer-key discrepancy.** The key prints $83.8\%$. Its $\omega_f = 3.125$ rad/s is right,
> but that $\omega_f$ gives $KE_f = \tfrac12(0.8)(3.125)^2 = 3.906$ J against $KE_i = 19.75$ J —
> a loss of $80.2\%$. The reduced-inertia formula independently returns the same $15.844$ J.
> Use **80.2 %**.

**What Python adds.** We show the *exact* structural parallel with the linear inelastic collision
(same formula, $m\to I$, $v\to\omega$) and derive the energy loss from the rotational reduced-mass
expression — then sweep $\omega_B$ to find the counter-rotation speed that dissipates **all** the
energy, i.e. brings both disks to a dead stop.

In [ ]:
# ═══ W10 · L2 · P6 — Rotational inelastic "collision", and total-loss tuning ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
I_A, w_A = 0.50,  8.0      # kg*m^2, rad/s
I_B, w_B = 0.30, -5.0      # counter-rotating

# --- (a) PREDICT: L conservation ----------------------------------------
L   = I_A*w_A + I_B*w_B
w_f = L / (I_A + I_B)
print(f"L = I_A w_A + I_B w_B = {I_A*w_A:+.3f} + {I_B*w_B:+.3f} = {L:+.4f} kg*m^2/s")
print(f"(a) omega_f = L / (I_A + I_B) = {w_f:.4f} rad/s")
assert np.isclose((I_A + I_B)*w_f, L)

# --- (b) energy lost -----------------------------------------------------
KE_i = 0.5*I_A*w_A**2 + 0.5*I_B*w_B**2
KE_f = 0.5*(I_A + I_B)*w_f**2
frac = (KE_i - KE_f) / KE_i
print(f"\n(b) KE before = {KE_i:.4f} J")
print(f"    KE after  = {KE_f:.4f} J")
print(f"    lost      = {KE_i - KE_f:.4f} J  = {100*frac:.1f}% of the initial energy")

# --- VERIFY with the rotational reduced-'mass' formula ------------------
I_red = I_A*I_B / (I_A + I_B)
lost_formula = 0.5 * I_red * (w_A - w_B)**2
print(f"\n    check: (1/2) * I_red * (w_A - w_B)^2 = {lost_formula:.4f} J -> agrees")
print(f"    (exactly the linear formula with m -> I and v -> omega)")
assert np.isclose(KE_i - KE_f, lost_formula)

# --- SWEEP: which w_B stops both disks dead? ---------------------------
wBs   = np.linspace(-30, 15, 900)
wfs   = (I_A*w_A + I_B*wBs) / (I_A + I_B)
KEis  = 0.5*I_A*w_A**2 + 0.5*I_B*wBs**2
KEfs  = 0.5*(I_A + I_B)*wfs**2
fracs = (KEis - KEfs)/KEis

w_B_stop = -I_A*w_A/I_B                       # L = 0 exactly
print(f"\nTotal-loss condition L = 0 requires w_B = -I_A w_A / I_B = {w_B_stop:.4f} rad/s")
print(f"  at that speed omega_f = {(I_A*w_A + I_B*w_B_stop)/(I_A+I_B):.2e} rad/s "
      "-> both disks stop dead, and 100% of the energy becomes heat.")
assert abs((I_A*w_A + I_B*w_B_stop)/(I_A + I_B)) < 1e-12

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 3.8))
ax1.plot(wBs, wfs, color="#1565c0", lw=2.5)
ax1.axhline(0, c="k", lw=.8); ax1.axvline(w_B, ls=":", c="grey")
ax1.plot(w_B, w_f, "o", color="crimson", ms=9, zorder=5, label=f"this problem")
ax1.plot(w_B_stop, 0, "*", color="#2e7d32", ms=16, zorder=5, label="dead stop")
ax1.set_xlabel("$\\omega_B$ (rad/s)"); ax1.set_ylabel("$\\omega_f$ (rad/s)")
ax1.set_title("final speed is linear in $\\omega_B$"); ax1.grid(alpha=.3); ax1.legend(fontsize=8)

ax2.plot(wBs, 100*fracs, color="#e65100", lw=2.5)
ax2.axvline(w_B, ls=":", c="grey")
ax2.plot(w_B, 100*frac, "o", color="crimson", ms=9, zorder=5,
         label=f"{100*frac:.1f}% lost")
ax2.plot(w_B_stop, 100, "*", color="#2e7d32", ms=16, zorder=5, label="100% lost")
ax2.set_xlabel("$\\omega_B$ (rad/s)"); ax2.set_ylabel("energy lost (%)")
ax2.set_title("dissipation peaks when L cancels"); ax2.grid(alpha=.3); ax2.legend(fontsize=8)
plt.suptitle("W10 P6 — coupling two coaxial disks", y=1.03)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(w_f - 3.125) < 1e-9, "omega_f matches the key"
assert abs(100*frac - 80.22) < 0.05, f"loss = {100*frac}%"
print(f"\nANSWER-KEY NOTE: the key prints 83.8% lost; two independent routes both give "
      f"{100*frac:.1f}%.")
print(f"  KE_f / KE_i = {KE_f:.4f} / {KE_i:.4f} = {KE_f/KE_i:.4f}  ->  lost {100*frac:.2f}%")
print(f"\n[OK] omega_f = {w_f:.3f} rad/s (matches key); CORRECTED loss = {100*frac:.1f}% of KE")

## L3 · Challenge — P9: Turntable, Putty, and a Restoring Motor

> **Problem (Week_10.ipynb, L3 — P9).** A turntable ($I = 0.015$ kg·m²) rotates freely at
> $33\tfrac13$ RPM. A $0.020$ kg ring of putty lands at radius $0.12$ m and sticks. (a) Find the
> new angular velocity. (b) A motor then restores the original speed in $2.0$ s. Find the required
> torque and the work done by the motor.

**Diagram → Principle.** Two distinct phases with **different conserved quantities**. Phase 1:
the putty lands — no external torque, so $L$ is conserved (and energy is lost). Phase 2: the motor
applies an external torque — $L$ is *not* conserved, and we use the angular impulse–momentum
theorem plus the work–energy theorem.

**Equation.** $\omega_1 = \dfrac{I_0\omega_0}{I_0 + mr^2}$; then
$\tau = \dfrac{\Delta L}{\Delta t}$ and $W = \Delta KE$.

> **Assumption, stated explicitly.** $\tau = \Delta L/\Delta t$ by itself gives only the
> **average** torque over the $2.0$ s. To quote a single $\alpha$, a definite swept angle
> $\theta$, and $W = \tau\theta$, we must assume the motor delivers a **constant** torque
> throughout the restoration. The problem does not say so, so we say it. Note $W = \Delta KE$
> holds regardless; it is $W = \tau\theta$ and the constant-$\alpha$ trajectory that depend on
> the assumption. Under any other torque profile only $\tau_{\rm avg}$ is determined.

**Hand prediction.** $I_{\rm putty} = mr^2 = 2.88\times10^{-4}$ kg·m², a small perturbation.

**What Python adds.** Students routinely apply "$W = \tau\theta$" and "$W = \Delta KE$"
interchangeably and get different numbers here — because during phase 2 the motor spins the
turntable through a definite angle, and both routes must be reconciled carefully. We compute both,
show they agree, and lay out the two-phase energy audit explicitly.

In [ ]:
# ═══ W10 · L3 · P9 — Two phases, two different conservation rules ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
I0    = 0.015                       # kg*m^2, bare turntable
m, r  = 0.020, 0.12                 # putty ring
rpm0  = 100/3                       # 33 1/3 RPM
w0    = rpm0 * 2*np.pi / 60
dt    = 2.0                         # motor restore time

I_putty = m * r**2
I1      = I0 + I_putty
print(f"omega_0 = {rpm0:.4f} RPM = {w0:.5f} rad/s")
print(f"I_putty = m r^2 = {I_putty:.3e} kg*m^2  ({100*I_putty/I0:.2f}% of the turntable)")
print(f"I1      = {I1:.6f} kg*m^2")

# ── PHASE 1: putty lands. NO external torque -> L conserved ────────────
w1 = I0*w0 / I1
print(f"\nPHASE 1 (L conserved, energy is not)")
print(f"(a) omega_1 = I0 w0 / I1 = {w1:.5f} rad/s = {w1*60/(2*np.pi):.4f} RPM")
print(f"    slowed by {100*(1 - w1/w0):.2f}%")
KE0, KE1 = 0.5*I0*w0**2, 0.5*I1*w1**2
print(f"    KE {KE0*1e3:.4f} mJ -> {KE1*1e3:.4f} mJ, lost {1e3*(KE0-KE1):.4f} mJ "
      f"({100*(KE0-KE1)/KE0:.2f}%)")
assert np.isclose(I0*w0, I1*w1), "L must be conserved in phase 1"

# ── PHASE 2: motor restores w0. External torque -> L NOT conserved ─────
dL      = I1*w0 - I1*w1                      # angular impulse needed
tau     = dL / dt
alpha   = tau / I1
theta   = w1*dt + 0.5*alpha*dt**2            # angle swept while accelerating
W_tau   = tau * theta                        # work = torque x angle
W_dKE   = 0.5*I1*w0**2 - 0.5*I1*w1**2        # work = change in KE

print(f"\nPHASE 2 (motor drives it back to omega_0 in {dt:.1f} s)")
print("  ASSUMPTION: the motor torque is CONSTANT over these 2.0 s. Without that,")
print("  dL/dt gives only the AVERAGE torque and theta (hence W = tau*theta) is")
print("  undetermined. W = delta KE holds either way.")
print(f"(b) angular impulse dL = I1 (w0 - w1) = {dL:.6e} kg*m^2/s")
print(f"    torque tau = dL/dt (constant)     = {tau:.6e} N*m")
print(f"    angular acceleration alpha        = {alpha:.5f} rad/s^2")
print(f"    angle swept theta                 = {theta:.4f} rad "
      f"({theta/(2*np.pi):.3f} rev)")
print(f"\n    work, route 1  W = tau * theta   = {W_tau*1e3:.6f} mJ")
print(f"    work, route 2  W = delta KE      = {W_dKE*1e3:.6f} mJ")
print(f"    residual                          = {abs(W_tau - W_dKE):.2e} J -> they agree")
assert abs(W_tau - W_dKE) < 1e-12

# --- The full two-phase energy audit ------------------------------------
KE2 = 0.5*I1*w0**2
print(f"\nENERGY AUDIT (mJ)")
print(f"  start (bare table at w0)     {KE0*1e3:9.4f}")
print(f"  after putty lands            {KE1*1e3:9.4f}   (lost {1e3*(KE0-KE1):.4f} to the impact)")
print(f"  after the motor restores w0  {KE2*1e3:9.4f}   (motor added {W_dKE*1e3:.4f})")
print(f"  net vs the start             {1e3*(KE2-KE0):+9.4f}   "
      "(higher: the system now carries the putty too)")
assert KE2 > KE0 > KE1

# --- Plot the whole history ---------------------------------------------
t_pre  = np.linspace(-1.0, 0, 50)
t_post = np.linspace(0, dt, 300)
fig, ax = plt.subplots(figsize=(7.4, 3.9))
ax.plot(t_pre,  np.full_like(t_pre, w0),  color="#1565c0", lw=2.5, label="free spin")
ax.plot([0, 0], [w0, w1], color="crimson", lw=2.5, ls=":", label="putty lands (instant)")
ax.plot(t_post, w1 + alpha*t_post, color="#2e7d32", lw=2.5, label="motor restoring")
ax.axhline(w0, ls="--", c="grey", lw=1)
ax.set_xlabel("t (s)"); ax.set_ylabel("$\\omega$ (rad/s)")
ax.set_title("W10 P9 — L conserved on the left of t = 0, torque-driven on the right")
ax.grid(alpha=.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(w1 - 3.42489) < 1e-4, f"w1 = {w1}"
assert abs(tau - 5.0265e-4) < 1e-6, f"tau = {tau}"

# A neat simplification worth noticing: dL = (I1 - I0) w0 = I_putty * w0 exactly.
print(f"\n  shortcut: dL = I1(w0 - w1) = (I1 - I0) w0 = I_putty * w0 = "
      f"{I_putty*w0:.6e} kg*m^2/s")
assert abs(dL - I_putty*w0) < 1e-15
print(f"\n[OK] omega_1 = {w1:.4f} rad/s ({w1*60/(2*np.pi):.2f} RPM); "
      f"tau = {tau:.3e} N*m; W = {W_dKE*1e3:.4f} mJ")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_10.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
